# DTAT399C. deeptrack.backend.units

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/units.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the `backend.units.py` module.

## 1. What is `units.py`?

The `units.py` module defines functions and a class to handle unit conversions. 

The `units.py` module is a foundational component of DeepTrack2’s unit-handling
infrastructure. It provides robust mechanisms to ensure physical quantities are
used correctly and consistently throughout simulations and models. In particular, this is useful when switching between optical and simulation domains.

Key Roles of `units.py`:

- **Unit Management and Standardization**  
  Provides a central interface to retrieve and manipulate physical units using
  the Pint library. This ensures that all numerical values in DeepTrack can be
  consistently interpreted and converted across simulation and optical domains.

- **Voxel Size and Pixel Scale Handling**  
  Implements utility functions such as `get_active_voxel_size()` and
  `get_active_scale()` that extract the currently active spatial resolution and
  scaling factors from the unit registry—critical for defining the physical
  meaning of pixels in space and time.

- **Dynamic Unit Context Creation**  
  Defines `create_context()`, a flexible utility for building Pint `Context`
  objects that map custom pixel sizes and simulation scale factors to physical
  units. This allows simulations to be configured with domain-specific unit
  behavior.

- **Batch Conversion with `ConversionTable`**  
  Provides a utility class that converts scalar and array-like values (including
  NumPy arrays and PyTorch tensors) from default to desired units using
  customizable mappings. It supports mixed data types and simplifies unit
  enforcement across pipelines.

- **Torch-Compatible Quantities**  
  Supports unit conversion of PyTorch tensors—enabling seamless integration of
  unit-aware logic in deep learning models, without sacrificing tensor
  operations or GPU compatibility.

- **Precision and Provenance in Physical Modeling**  
  Encourages reproducibility and clarity in simulations by embedding units
  explicitly in data processing steps, reducing the risk of hidden assumptions
  or mismatches in scale.

## 2.  Retrieving Voxel Size and Pixel Scale

The voxel size and pixel scale are fundamental for interpreting simulation data
in physical units. DeepTrack2 uses `pint` to define and convert these quantities
via a centralized unit registry.

### 2.1. Retrieving the Voxel Size

This gives the physical size (in meters) of a simulation voxel along each axis:

In [2]:
from deeptrack.backend import units

voxel_size = units.get_active_voxel_size()

print("Voxel size (x, y, z) in meters:", voxel_size)

Voxel size (x, y, z) in meters: (1e-06, 1e-06, 1e-06)


This means each voxel is 1 micron (1e-6 meters) in each spatial direction.

### 2.2. Retrieving the Scale Between Optical and Simulation Pixels

This gives the scaling factor between optical and simulation pixel units:

In [3]:
scale = units.get_active_scale()

print("Pixel scale factors (x, y, z):", scale)

Pixel scale factors (x, y, z): (1.0, 1.0, 1.0)


A scale of 1.0 means there's no up/downsampling between optical and simulation space.
These scaling factors can change with simulation settings or microscope calibration.

## 3. Defining Custom Units

You can override the default voxel sizes and simulation scaling using a unit context.
This is useful when adapting to different microscope setups or simulation resolutions.

Define a unit context:

In [4]:
ctx = units.create_context(
    xpixel=2e-6,     # 2 microns per optical x-pixel
    ypixel=1e-6,     # 1 micron per optical y-pixel
    zpixel=0.5e-6,   # 0.5 microns per optical z-pixel
    xscale=2,        # 2× upscaling in simulation along x
    yscale=1,        # no change along y
    zscale=4         # 4× upscaling in simulation along z
)

Once defined, the context must be activated using the unit registry. This allows
you to convert simulation pixels into actual distances.

In [5]:
from deeptrack import units_registry as u

with u.context(ctx):
    print("1 simulation x-pixel =", (1 * u.simulation_xpixel).to("meter"))
    print("1 simulation z-pixel =", (1 * u.simulation_zpixel).to("meter"))

1 simulation x-pixel = 1e-06 meter
1 simulation z-pixel = 1.25e-07 meter


Why? Because:

* simulation_xpixel = xpixel / xscale = 2e-6 / 2 = 1e-6

* simulation_zpixel = zpixel / zscale = 0.5e-6 / 4 = 1.25e-7

## 4. Converting Values to Target Units

The `ConversionTable` allows you to apply unit conversions to dictionaries of values
— useful in data preprocessing pipelines or during model I/O.

Define a conversion table:

In [6]:
conversion = units.ConversionTable(
    length=(u.meter, u.micrometer),
    time=(u.second, u.millisecond),
    velocity=(u.meter / u.second, u.kilometer / u.hour),
    temperature=(u.kelvin, u.celsius),
)

### 4.1. Converting scalar values

In [7]:
converted = conversion.convert(
    length=1.0,
    time=0.5,
    velocity=10,
    temperature=0,
    NON_EXISTENT_VALUE=100,  # Not converted
)

converted

{'length': 1000000.0 <Unit('micrometer')>,
 'time': 500.0 <Unit('millisecond')>,
 'velocity': 36.0 <Unit('kilometer / hour')>,
 'temperature': -273.15 <Unit('degree_Celsius')>,
 'NON_EXISTENT_VALUE': 100}

This can be also written equivalently as:

In [8]:
values = {
    "length": 1.0,
    "time": 0.5,
    "velocity": 10,
    "temperature": 0,
    "NON-EXISTENT-VALUE": 100,  # Not converted
}

converted = conversion.convert(**values)

converted

{'length': 1000000.0 <Unit('micrometer')>,
 'time': 500.0 <Unit('millisecond')>,
 'velocity': 36.0 <Unit('kilometer / hour')>,
 'temperature': -273.15 <Unit('degree_Celsius')>,
 'NON-EXISTENT-VALUE': 100}

### 4.2. Converting NumPy arrays

In [9]:
import numpy as np

array = np.array([1.0, 2.0])

converted = conversion.convert(length=array)

converted["length"]

Magnitude,[1000000.0 2000000.0]
Units,micrometer


### 4.3. Convert Torch Tensors

In [10]:
import torch

tensor = torch.tensor([1.0, 2.0])

converted = conversion.convert(length=tensor)

converted["length"]


Magnitude,"tensor([1000000., 2000000.])"
Units,micrometer


## 5. Direct Use with Pint Quantities

The `units_registry` also allows direct conversion of raw quantities without
defining contexts or tables. This is useful for ad-hoc conversions and debugging unit mismatches.

Import the unit regustry:

In [11]:
from deeptrack import units_registry as u

In [12]:
(1 * u.meter).to("micrometer")

1000000.0 <Unit('micrometer')>

In [13]:
print((2 * u.second).to("millisecond"))

2000.0 millisecond
